<a href="https://colab.research.google.com/github/tannishgarg01/ACT-104-Week-4-/blob/main/ACT_104_Week_4_(Tannish).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment: Tool‑Augmented LLM App (Weather + Calculator)

**Goal:** Build a minimal GenAI pipeline that uses **memory/caching** and **two tools** (weather + calculator), with a simple agent that routes to tools when needed.



## Instructions

- Implement the required functions in the **solution cells**
- Use a module‑level cache created by `setup_cache()`; repeated `(city, unit)` queries should hit the cache (no re‑fetch).
- Keep your code deterministic for grading.
- You may integrate LangChain locally, but the autograder focuses on tool behavior + caching and a minimal agent router.
- **Do not rename** required functions or change their signatures.

### Required functions

1. `setup_cache() -> Any`
2. `get_weather(city: str, unit: str = "C", fetcher: callable | None = None) -> dict`
3. `multiply(a: float, b: float) -> float`
4. `tool_registry() -> dict`
5. `agent_route(prompt: str) -> str`  (return `"weather"`, `"multiply"`, or `"none"`)
6. `agent_answer(prompt: str, tools: dict | None = None, cache=None) -> str`

### Playtests (not graded)
Try these prompts after you finish:
- “What’s the weather in Paris?”
- “What’s 42 × 19?”
- “Summarize both results in one sentence.”


In [ ]:

# Read-Only imports (you may import more inside solutions)
from typing import Any, Callable, Dict, Optional
import re
import requests  # allowed for tests


## OpenRouter API Setup

This assignment is configured to use **OpenRouter** instead of provider-specific API keys. Enter your OpenRouter API key when prompted. Do not hard-code the key or save it in the notebook.

The starter assignment and grading logic focus on the required tool, caching, and routing functions. The OpenRouter configuration is provided so any optional LLM calls can use the same key.

In [ ]:
import os
from getpass import getpass

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("Enter your OpenRouter API key: ")

OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
print("OpenRouter API key configured.")


Enter your OpenRouter API key: ··········
OpenRouter API key configured.



### Q1. Implement `setup_cache()` *(Solution cell)*
Create a cache object (e.g., dict) and assign it to module-level `CACHE`. Return it.


In [ ]:
CACHE: Any = None

def setup_cache() -> Any:
    """Initialize and assign a dictionary cache to module-level CACHE."""
    global CACHE
    CACHE = {}
    return CACHE

---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  Make sure to save your function in the variable named `setup_cache`.
 -  If the function is not found, you answer will be rejected by the system

In [ ]:
# BEGIN PUBLIC TESTS
c = setup_cache()
assert c is not None and isinstance(c, dict)
assert 'CACHE' in globals() and isinstance(CACHE, dict)
# END PUBLIC TESTS


### Q2. Implement `get_weather(city, unit="C")` *(Solution cell)*
- Use Open-Meteo Geocoding to resolve city → (lat, lon).
- Call Open-Meteo Current Weather for temperature (Celsius).
- If `unit == "F"`, convert Celsius to Fahrenheit.
- Cache result dict as `{"city": city, "temp": <float>, "unit": unit}` under normalized key.
- Subsequent calls with same `(city, unit)` must return the **same dict object** from cache (so `is` equality works).


In [ ]:
def get_weather(
    city: str,
    unit: str = "C",
    fetcher: Optional[Callable] = None
) -> dict:
    """
    Return current weather for a city, with normalized (city, unit) caching.

    If `fetcher` is supplied, it is used instead of making network calls.
    A custom fetcher may return a Celsius float/int, a dict containing
    `temp`/`temperature`, or a requests-like response object.
    """
    global CACHE

    if CACHE is None:
        setup_cache()

    city_clean = str(city).strip()
    unit_clean = str(unit).strip().upper()

    if not city_clean:
        raise ValueError("city must not be empty")
    if unit_clean not in {"C", "F"}:
        raise ValueError("unit must be 'C' or 'F'")

    key = (city_clean.casefold(), unit_clean)

    # Important for the assignment: return the exact same object on cache hit.
    if key in CACHE:
        return CACHE[key]

    # Optional fetcher makes the function deterministic and easy to test.
    if fetcher is not None:
        raw = fetcher(city_clean)

        if isinstance(raw, (int, float)):
            temp_c = float(raw)
        elif isinstance(raw, dict):
            if "temp_c" in raw:
                temp_c = float(raw["temp_c"])
            elif "temperature" in raw:
                temp_c = float(raw["temperature"])
            elif "temp" in raw:
                temp_c = float(raw["temp"])
            else:
                raise ValueError("fetcher dict must contain temp_c, temperature, or temp")
        elif hasattr(raw, "json"):
            data = raw.json()
            if "current_weather" in data:
                temp_c = float(data["current_weather"]["temperature"])
            elif "temperature" in data:
                temp_c = float(data["temperature"])
            else:
                raise ValueError("fetcher response did not contain a temperature")
        else:
            raise ValueError("Unsupported fetcher return value")
    else:
        # 1) City -> coordinates
        geo = requests.get(
            "https://geocoding-api.open-meteo.com/v1/search",
            params={"name": city_clean, "count": 1, "language": "en", "format": "json"},
            timeout=10,
        )
        geo.raise_for_status()
        geo_data = geo.json()

        if not geo_data.get("results"):
            raise ValueError(f"City not found: {city_clean}")

        location = geo_data["results"][0]
        lat = location["latitude"]
        lon = location["longitude"]

        # 2) Coordinates -> current temperature in Celsius
        weather = requests.get(
            "https://api.open-meteo.com/v1/forecast",
            params={"latitude": lat, "longitude": lon, "current_weather": "true"},
            timeout=10,
        )
        weather.raise_for_status()
        weather_data = weather.json()
        temp_c = float(weather_data["current_weather"]["temperature"])

    temp = temp_c if unit_clean == "C" else (temp_c * 9.0 / 5.0) + 32.0

    result = {
        "city": city_clean,
        "temp": float(temp),
        "unit": unit_clean,
    }

    CACHE[key] = result
    return result

---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  Make sure to save your function in the variable named `get_weather`.
 -  If the function is not found, you answer will be rejected by the system

In [ ]:
# Q2 — Public Tests (API call allowed)
# BEGIN PUBLIC TESTS
setup_cache()
d1 = get_weather("Paris", "C")
d2 = get_weather("paris", "c")
assert d1 is d2, "Cached object should be returned for repeated queries."
assert isinstance(d1["temp"], float) and d1["unit"] == "C"
# Fahrenheit conversion
d3 = get_weather("Paris", "F")
assert d3["unit"] == "F" and isinstance(d3["temp"], float)
# END PUBLIC TESTS


### Q3. Implement `multiply(a, b)` *(Solution cell)*
Return the product of `a` and `b`.


In [ ]:
def multiply(a: float, b: float) -> float:
    """Return the product of two numbers."""
    return float(a) * float(b)

---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  Make sure to save your function in the variable named `multiply`.
 -  If the function is not found, you answer will be rejected by the system

In [ ]:
# Q3 — Public Tests
# BEGIN PUBLIC TESTS
assert multiply(6, 7) == 42
assert abs(multiply(1.5, 2.0) - 3.0) < 1e-9
# END PUBLIC TESTS

In [ ]:
# Q3 — Hidden Tests


### Q4. Implement `tool_registry()` *(Solution cell)*
Return a dict mapping `{"weather": get_weather, "multiply": multiply}`.


In [ ]:
def tool_registry() -> Dict[str, Callable]:
    """Return the tools exposed to the agent router."""
    return {
        "weather": get_weather,
        "multiply": multiply,
    }

---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  Make sure to save your function in the variable named `tool_registry`.
 -  If the function is not found, you answer will be rejected by the system

In [ ]:
# Q4 — Public Tests
# BEGIN PUBLIC TESTS
reg = tool_registry()
assert isinstance(reg, dict)
assert "weather" in reg and "multiply" in reg
assert callable(reg["weather"]) and callable(reg["multiply"])
# END PUBLIC TESTS

In [ ]:
# Q4 — Hidden Tests


### Q5. Implement `agent_route(prompt)` *(Solution cell)*
Return `"weather"`, `"multiply"`, or `"none"` based on the prompt.


In [ ]:
def agent_route(prompt: str) -> str:
    """
    Select a tool based on the user's prompt.

    Returns exactly: "weather", "multiply", or "none".
    """
    text = str(prompt).strip().lower()

    # Weather intent: explicit weather/temperature requests.
    weather_pattern = r"\b(weather|temperature|forecast)\b"
    if re.search(weather_pattern, text):
        return "weather"

    # Multiplication intent: x, ×, *, "multiply", "times", or "product".
    multiply_pattern = (
        r"(?:\d+(?:\.\d+)?\s*[x×*]\s*[-+]?\d+(?:\.\d+)?)"
        r"|(?:multiply|times|product)\b"
    )
    if re.search(multiply_pattern, text):
        return "multiply"

    return "none"

---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  Make sure to save your function in the variable named `agent_route`.
 -  If the function is not found, you answer will be rejected by the system

In [ ]:
# Q5 — Public Tests
# BEGIN PUBLIC TESTS
assert agent_route("what's the weather in Jaipur?") == "weather"
assert agent_route("compute 42 x 19 please") == "multiply"
assert agent_route("say hello") == "none"
# END PUBLIC TESTS


In [ ]:
# Q5 — Hidden Tests


### Q6. Implement `agent_answer(prompt, tools=None, cache=None)` *(Solution cell)*
- Use `agent_route` to choose a tool.
- If weather: parse city from `"weather in <City>"`. Default unit `"C"`, but if prompt mentions "Fahrenheit" or "F", use `"F"`.
- If multiply: parse two numbers from `a x b` or `a * b` and compute result with `multiply`.
- Return a short string response. You may pass `cache` in to override the module cache.


In [ ]:
def agent_answer(
    prompt: str,
    tools: dict | None = None,
    cache=None
) -> str:
    """
    Route the prompt to a required tool and return a short textual answer.
    """
    if tools is None:
        tools = tool_registry()

    route = agent_route(prompt)

    if route == "weather":
        # Parse "weather in <city>" and remove optional unit phrases.
        match = re.search(
            r"\b(?:weather|temperature|forecast)\s+in\s+(.+?)(?:\s+in\s+"
            r"(?:fahrenheit|celsius|f|c))?\s*[?!.]*$",
            str(prompt),
            flags=re.IGNORECASE,
        )

        if not match:
            return "Please provide a city, for example: weather in Paris."

        city = match.group(1).strip(" ,?.!")
        unit = "F" if re.search(r"\b(?:fahrenheit|F)\b", str(prompt), re.IGNORECASE) else "C"

        # agent_answer may receive an explicit cache. get_weather uses the
        # module-level cache, so temporarily route it to the supplied cache.
        global CACHE
        old_cache = CACHE
        if cache is not None:
            CACHE = cache

        try:
            data = tools["weather"](city, unit)
        finally:
            if cache is not None:
                CACHE = old_cache

        return f"Weather in {data['city']}: {data['temp']} {data['unit']}"

    if route == "multiply":
        # Supports "12 * 8", "12 x 8", and "12 × 8".
        match = re.search(
            r"([-+]?\d+(?:\.\d+)?)\s*[x×*]\s*([-+]?\d+(?:\.\d+)?)",
            str(prompt),
            flags=re.IGNORECASE,
        )

        # Also support natural language such as "multiply 12 by 8".
        if not match:
            match = re.search(
                r"(?:multiply|times|product\s+of)\s+"
                r"([-+]?\d+(?:\.\d+)?)\s+(?:by|and|times)\s+"
                r"([-+]?\d+(?:\.\d+)?)",
                str(prompt),
                flags=re.IGNORECASE,
            )

        if not match:
            return "Please provide two numbers to multiply."

        a = float(match.group(1))
        b = float(match.group(2))
        result = tools["multiply"](a, b)
        return f"{a} * {b} = {result}"

    return "I can answer directly without using a tool."

---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  Make sure to save your function in the variable named `agent_answer`.
 -  If the function is not found, you answer will be rejected by the system

In [ ]:

# Q6 — Public Tests
# BEGIN PUBLIC TESTS
cache = setup_cache()
tools = tool_registry()

o1 = agent_answer("What's the weather in Mumbai in Fahrenheit?", tools=tools, cache=cache)
assert "Mumbai" in o1 and ("F" in o1)

o2 = agent_answer("Please do 12 * 8", tools=tools, cache=cache)
assert "12.0 * 8.0 = 96.0" in o2 or "12 * 8 = 96" in o2
# END PUBLIC TESTS


In [ ]:

# Q6 — Hidden Tests


## Playtest Logs — Vanilla vs Tool-Augmented

The required functions above implement the graded tool behavior. The following optional cells demonstrate the three requested playtests.

In [ ]:
# Three required-style playtests.
cache = setup_cache()
tools = tool_registry()

print("1.", agent_answer("What's the weather in Paris?", tools=tools, cache=cache))
print("2.", agent_answer("What's 42 × 19?", tools=tools, cache=cache))
print("3.", agent_answer(
    "Summarize both results in one sentence.",
    tools=tools,
    cache=cache
))

1. Weather in Paris: 22.9 C
2. 42.0 * 19.0 = 798.0
3. I can answer directly without using a tool.


## Reflection — 500–700 words

Tool use extends an LLM application by giving the model access to capabilities that are not reliably available through prompting alone. In this assignment, the application has two external tools: a weather function that retrieves current temperature data from Open-Meteo and a multiplication function that performs an exact arithmetic operation. Instead of asking a language model to guess a weather value or calculate a result itself, the application can route the request to a deterministic tool and then present the returned result to the user.

The weather tool demonstrates why external APIs are useful. Weather information changes over time, so a model's stored knowledge is not an appropriate source for current conditions. The tool first resolves a city to coordinates using geocoding and then requests current weather data. The calculator tool demonstrates a different advantage: deterministic computation. Multiplication can be performed exactly by code, avoiding unnecessary reliance on probabilistic text generation.

Memory and caching improve efficiency by avoiding repeated external calls for the same request. The implementation normalizes the city and unit before constructing a cache key, so requests such as “Paris, C” and “paris, c” can use the same cached object. This reduces repeated network traffic and can lower latency and API usage. The trade-off is that cached data can become stale, especially for rapidly changing information such as weather. A production system would therefore need a time-to-live policy or another invalidation strategy.

The agent router provides a simple form of orchestration. It examines the user's prompt and selects the weather tool, multiplication tool, or no tool. This is intentionally lightweight and deterministic because the grading environment focuses on routing and tool behavior. A more advanced application could expose tool schemas to an LLM and use structured function calling, allowing the model to select tools and provide arguments in a structured format.

There are several trade-offs. Tool-augmented systems are more capable, but they are also more complex than a direct LLM call. Developers must handle API failures, invalid inputs, timeouts, authentication, caching, parsing, and tool results. Tool use can also add latency because the application may need one or more network requests before generating a response. On the positive side, the system becomes more transparent: a developer can inspect which tool was selected and verify the returned data.

I would extend the system by using hybrid caching. Frequently requested stable results could have longer cache lifetimes, while weather data could use a short time-to-live. Multiple APIs could also be combined with fallback logic, so a second provider is used if the first one is unavailable. For more complex tasks, function-chaining could allow the output of one tool to become the input of another. For example, an application could retrieve weather, calculate a unit conversion, and then produce a concise recommendation.

Overall, the assignment showed that an LLM does not need to perform every task itself. Tools provide reliable external capabilities, caching reduces repeated work, and orchestration determines when those capabilities should be used. Combining these components makes an application more useful, but it also introduces engineering responsibilities that do not exist in a simple prompt-and-response system.